# 🛣️ Road Damage & Pothole Segmentation using SAM (Segment Anything Model)

Notebook ini mengimplementasikan segmentasi kerusakan jalan (**Pothole, Crack, Manhole, Grate**) menggunakan arsitektur **Segment Anything Model (SAM)** dan **Hybrid Pipeline (YOLO + SAM)**.

### 🌟 Fitur Utama:
1. **Hybrid Pipeline (YOLO Detection + SAM Mask Refiner)**: Menggunakan YOLO untuk deteksi cepat lokasi kerusakan, kemudian di-refine oleh SAM untuk menghasilkan kontur mask beresolusi super presisi (mengatasi masalah retakan tipis/tepi kasar).
2. **Prompt-based SAM**: Segmentasi berbasis titik (*points*) atau bounding box langsung ke area jalan berlubang/retak.
3. **Automatic Road Masking**: Menemukan seluruh pola objek & retakan di permukaan jalan secara otomatis.
4. **Kalkulasi Luas Kerusakan**: Menghitung estimasi luas area piksel kerusakan jalan untuk analisis keparahan (*damage severity assessment*).

## 1. Setup Environment & Import Libraries
Memuat dependensi yang dibutuhkan (`ultralytics`, `opencv`, `matplotlib`, `torch`).

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Ultralytics mendukung SAM (SAM 1, SAM 2, FastSAM, MobileSAM)
import ultralytics
from ultralytics import YOLO, SAM, FastSAM

print(f"Ultralytics version : {ultralytics.__version__}")
print(f"CUDA Available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total           : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


## 2. Load Model SAM & YOLO Detector

Pilihan model SAM yang kompatibel dengan GPU RTX 3050 (4GB VRAM):
- `sam2_t.pt` (SAM 2 Tiny) — Sangat cepat & efisien VRAM (~1.5GB VRAM)
- `sam2_s.pt` (SAM 2 Small) — Keseimbangan ideal antara ketajaman mask & kecepatan
- `FastSAM-s.pt` (FastSAM) — Sangat cepat untuk inferensi real-time

In [ ]:
# 1. Load Model SAM (Pilih salah satu)
SAM_MODEL_NAME = "sam2_t.pt"   # atau 'sam2_s.pt' / 'FastSAM-s.pt'
print(f"Loading SAM model: {SAM_MODEL_NAME}...")
sam_model = SAM(SAM_MODEL_NAME)

# 2. Load Model YOLO (sebagai detector pembantu bounding box)
# Gunakan model terbaik hasil training Anda atau model pretrained
yolo_weights_path = Path(r"runs/segment/runs/segment/pothole_seg_v1/weights/best.pt")
if not yolo_weights_path.exists():
    yolo_weights_path = "yolo26s-seg.pt"  # fallback

print(f"Loading YOLO model: {yolo_weights_path}...")
yolo_model = YOLO(str(yolo_weights_path))
print("✅ Kedua model berhasil dimuat!")


## 3. Mode A: Hybrid Pipeline (YOLO Detection ➡️ SAM Mask Refinement)

**Cara kerja:**
1. YOLO mendeteksi bounding box dan mengklasifikasikan kelas (`pothole`, `crack`, `manhole`, `grate`).
2. Setiap bounding box diteruskan ke SAM sebagai *Box Prompt*.
3. SAM menghasilkan kontur poligon berkualitas ultra-tinggi yang presisi mengikuti lekukan retakan/lubang jalan.

In [ ]:
def run_hybrid_yolo_sam(image_path, yolo_net, sam_net, conf=0.25, iou=0.5):
    """
    Menjalankan deteksi kotak dengan YOLO lalu menghasilkan mask berpresisi tinggi dengan SAM.
    """
    img = cv2.imread(str(image_path))
    if img is None:
        raise IOError(f"Gambar tidak ditemukan: {image_path}")
    
    h, w = img.shape[:2]
    
    # 1. Step 1: Deteksi Bounding Box menggunakan YOLO
    yolo_res = yolo_net.predict(img, conf=conf, iou=iou, imgsz=800, verbose=False)[0]
    
    if len(yolo_res.boxes) == 0:
        print("⚠️ Tidak ada objek kerusakan yang terdeteksi oleh YOLO.")
        return img, yolo_res, None
    
    boxes_xyxy = yolo_res.boxes.xyxy.cpu().numpy()  # [N, 4]
    cls_ids = yolo_res.boxes.cls.cpu().numpy().astype(int)
    confs = yolo_res.boxes.conf.cpu().numpy()
    
    # Terjemahan nama kelas
    class_names_map = {
        "bache": "pothole",
        "grieta": "crack",
        "registro": "manhole",
        "rejilla": "drainage grate"
    }
    
    # 2. Step 2: Kirim Bounding Box ke SAM sebagai Prompt
    sam_res = sam_net.predict(
        source=img,
        bboxes=boxes_xyxy.tolist(),
        retina_masks=True,
        device=0,
        verbose=False
    )[0]
    
    # 3. Step 3: Visualisasi Gabungan (Overlay SAM Mask + Label YOLO)
    overlay = img.copy()
    colors = [
        (0, 0, 255),    # Merah : Pothole
        (255, 0, 0),    # Biru : Crack
        (0, 255, 0),    # Hijau : Manhole
        (0, 255, 255)   # Kuning : Grate
    ]
    
    if sam_res.masks is not None:
        masks_np = sam_res.masks.data.cpu().numpy() # [N, H, W]
        for i in range(len(masks_np)):
            mask_i = (masks_np[i] > 0.5).astype(np.uint8)
            # Resize jika resolusi mask berbeda dengan gambar asli
            if mask_i.shape != (h, w):
                mask_i = cv2.resize(mask_i, (w, h), interpolation=cv2.INTER_NEAREST)
            
            cls_id = cls_ids[i] if i < len(cls_ids) else 0
            raw_name = yolo_net.names.get(cls_id, f"cls_{cls_id}")
            cls_name = class_names_map.get(raw_name, raw_name)
            conf_val = confs[i] if i < len(confs) else 1.0
            color = colors[cls_id % len(colors)]
            
            # Beri warna pada mask
            colored_mask = np.zeros_like(img, dtype=np.uint8)
            colored_mask[mask_i == 1] = color
            overlay = cv2.addWeighted(overlay, 1.0, colored_mask, 0.45, 0)
            
            # Gambar kontur tepi mask
            contours, _ = cv2.findContours(mask_i, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(overlay, contours, -1, color, 2)
            
            # Gambar bounding box dan label
            x1, y1, x2, y2 = map(int, boxes_xyxy[i])
            cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)
            label_text = f"{cls_name} {conf_val:.2f}"
            cv2.putText(overlay, label_text, (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            
    return overlay, yolo_res, sam_res

print("✅ Fungsi Hybrid Pipeline siap digunakan!")


### Jalankan Hybrid Pipeline & Bandingkan Hasilnya
Mari kita uji pada gambar jalan `testimage3.png` dan bandingkan hasil segmentasi murni YOLO vs Hybrid (YOLO + SAM).

In [ ]:
IMAGE_TEST = "testimage3.png"   # Ganti dengan 'testimage.png', 'testimage2.png', dll.

# Jalankan Hybrid YOLO + SAM
hybrid_result, yolo_out, sam_out = run_hybrid_yolo_sam(
    image_path=IMAGE_TEST,
    yolo_net=yolo_model,
    sam_net=sam_model,
    conf=0.20
)

# Visualisasi Komparasi Side-by-Side
img_orig = cv2.cvtColor(cv2.imread(IMAGE_TEST), cv2.COLOR_BGR2RGB)
yolo_plot = cv2.cvtColor(yolo_out.plot(), cv2.COLOR_BGR2RGB)
hybrid_plot = cv2.cvtColor(hybrid_result, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(img_orig)
axes[0].set_title("Gambar Asli Jalan", fontsize=12)
axes[0].axis("off")

axes[1].imshow(yolo_plot)
axes[1].set_title("Hasil YOLO Standar (Mask Kasar)", fontsize=12)
axes[1].axis("off")

axes[2].imshow(hybrid_plot)
axes[2].set_title("Hasil Hybrid YOLO + SAM (Mask Presisi Tinggi)", fontsize=12)
axes[2].axis("off")

plt.tight_layout()
plt.show()

# Simpan hasil
cv2.imwrite("hasil_hybrid_sam.jpg", hybrid_result)
print("✅ Hasil hybrid disimpan sebagai 'hasil_hybrid_sam.jpg'")


## 4. Mode B: Prompt-Based SAM (Point & Box Prompting)

Anda dapat memberikan koordinat titik (*point prompt*) atau koordinat kotak tertentu untuk mengisolasi lubang/retakan jalan secara interaktif.

In [ ]:
# Contoh: Memberikan Box Prompt langsung ke SAM
IMAGE_SAMPLE = "testimage3.png"
img_bgr = cv2.imread(IMAGE_SAMPLE)
h, w = img_bgr.shape[:2]

# Definisikan koordinat kotak area jalan yang dicurigai berlubang [x1, y1, x2, y2]
# Contoh box area tengah-bawah gambar jalan:
custom_box = [[int(w * 0.2), int(h * 0.3), int(w * 0.8), int(h * 0.9)]]

results_prompt = sam_model.predict(
    source=IMAGE_SAMPLE,
    bboxes=custom_box,
    device=0,
    retina_masks=True
)

annotated_prompt = results_prompt[0].plot()
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(annotated_prompt, cv2.COLOR_BGR2RGB))
plt.title("SAM Prompt-based Segmentation Result")
plt.axis("off")
plt.show()


## 5. Mode C: Automatic Road Defect Masking (Segment Everything)

SAM memindai seluruh grid gambar dan membuat segmentasi untuk semua struktur/tekstur yang berbeda di jalan tanpa memerlukan anotasi awal.

In [ ]:
# Segment Everything Mode
auto_results = sam_model.predict(
    source="testimage3.png",
    device=0,
    retina_masks=True,
    conf=0.25
)

auto_plot = auto_results[0].plot()
plt.figure(figsize=(10, 7))
plt.imshow(cv2.cvtColor(auto_plot, cv2.COLOR_BGR2RGB))
plt.title(f"SAM Auto-Segment Everything ({len(auto_results[0].masks) if auto_results[0].masks is not None else 0} segment masks)")
plt.axis("off")
plt.show()


## 6. Analisis Keparahan Kerusakan (Damage Severity & Area Estimation)

Fitur untuk riset **Hibah VLM**: Menghitung luas piksel area jalan yang rusak dan rasio persentase kerusakan terhadap keseluruhan jalan.

In [ ]:
def calculate_damage_metrics(sam_results, image_shape):
    """
    Menghitung total area piksel kerusakan jalan dan estimasi persentase luasnya.
    """
    h, w = image_shape[:2]
    total_road_pixels = h * w
    
    if sam_results.masks is None:
        return 0, 0.0, "Kondisi Jalan Baik (Tidak Terdeteksi Kerusakan)"
    
    masks_data = sam_results.masks.data.cpu().numpy()
    combined_mask = np.zeros((h, w), dtype=np.uint8)
    
    for m in masks_data:
        m_binary = (m > 0.5).astype(np.uint8)
        if m_binary.shape != (h, w):
            m_binary = cv2.resize(m_binary, (w, h), interpolation=cv2.INTER_NEAREST)
        combined_mask = np.bitwise_or(combined_mask, m_binary)
        
    damage_pixels = int(np.sum(combined_mask))
    damage_ratio = (damage_pixels / total_road_pixels) * 100
    
    if damage_ratio < 1.0:
        severity = "Ringan (Minor Damage)"
    elif damage_ratio < 5.0:
        severity = "Sedang (Moderate Damage)"
    else:
        severity = "Parah / Kritis (Severe Road Damage)"
        
    return damage_pixels, damage_ratio, severity

if sam_out is not None:
    dmg_px, dmg_pct, sev = calculate_damage_metrics(sam_out, img_orig.shape)
    print("=" * 50)
    print("📊 LAPORAN KEPARAHAN KERUSAKAN JALAN (VLM METRICS)")
    print("=" * 50)
    print(f"Total Luas Gambar       : {img_orig.shape[1]} x {img_orig.shape[0]} px ({img_orig.shape[0]*img_orig.shape[1]:,} px)")
    print(f"Luas Area Kerusakan     : {dmg_px:,} piksel")
    print(f"Rasio Kerusakan Jalan   : {dmg_pct:.2f}%")
    print(f"Status Keparahan        : {sev}")
    print("=" * 50)


## 7. Video Real-Time Road Inspection dengan FastSAM / SAM
Menjalankan segmentasi pada file video jalan (`testvideo1.mp4`).

In [ ]:
import cv2

VIDEO_INPUT = "testvideo1.mp4"   # atau 'testvideo2.mp4'
OUTPUT_VIDEO = "hasil_video_sam.mp4"

cap = cv2.VideoCapture(VIDEO_INPUT)
if not cap.isOpened():
    print(f"⚠️ Video {VIDEO_INPUT} tidak ditemukan, lewati langkah ini.")
else:
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (w, h))
    
    print(f"Memproses video {VIDEO_INPUT}... Tekan 'q' di jendela popup untuk berhenti lebih awal.")
    frame_count = 0
    
    # Kita proses 60 frame pertama sebagai demonstrasi cepat
    while cap.isOpened() and frame_count < 60:
        ret, frame = cap.read()
        if not ret:
            break
            
        # Jalankan hybrid YOLO+SAM per frame
        y_res = yolo_model.predict(frame, conf=0.3, imgsz=640, verbose=False)[0]
        if len(y_res.boxes) > 0:
            boxes = y_res.boxes.xyxy.cpu().numpy().tolist()
            s_res = sam_model.predict(frame, bboxes=boxes, device=0, verbose=False)[0]
            annotated = s_res.plot()
        else:
            annotated = frame
            
        out.write(annotated)
        frame_count += 1
        
    cap.release()
    out.release()
    print(f"✅ Pemrosesan video selesai! Disimpan ke: {OUTPUT_VIDEO}")
